# Baseline WER Test - Cleft Speech Correction Capstone (FIXED v3)Your last run improved a lot (WER dropped from 257.7% to 83.9%, and the gibberish/foreign-language/repeated-word problems are gone). The remaining issue is specific toyour SHORT single-word recordings, which were getting fabricated, fluent-but-wrongsentences as predictions.Two fixes in this version:1. Very short recordings (under 1.2 seconds) are now padded with a bit of silence   during conversion - this reduces Whisper's tendency to hallucinate on near-instant   audio.2. Decoding is now deterministic (temperature=0), which stops the model from   "creatively" filling in plausible-but-wrong sentences.IMPORTANT: use this file directly (Colab: File > Upload notebook), don't retype orcopy-paste the code elsewhere first.

## Step 1 - Prepare your files in Google DriveFolder structure in "My Drive":```capstone-cleft-speech-data/|-- raw_recordings/\-- tracking/     \-- dataset.csv```

## Step 2 - Install required tools

In [ ]:
!pip install -q transformers jiwer pydub librosa soundfile accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.0 MB/s eta 0:00:00


## Step 3 - Connect to your Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_path = "/content/drive/MyDrive/capstone-cleft-speech-data"
print("Folders found:", os.listdir(base_path))
print("Sample recordings:", os.listdir(f"{base_path}/raw_recordings")[:5])

Mounted at /content/drive
Folders found: ['raw_recordings', 'tracking', 'finetuned_whisper_model', 'Research', 'augmented_wav', 'finetuned_whisper_lora', 'converted_wav']
Sample recordings: ['vase2.mp3', 'elephant3.mp3', 'fish2.mp3', 'coffee.mp3', 'elephant2.mp3']


## Step 4 - Load your dataset.csv

In [ ]:
import pandas as pd

csv_path = f"{base_path}/tracking/dataset.csv"
df = pd.read_csv(csv_path)
print("Total rows:", len(df))
df.head()

Total rows: 360


,filename,intended_text
0,paper.mp3,paper
1,paper2.mp3,paper
2,paper3.mp3,paper
3,pencil.mp3,pencil
4,pencil2.mp3,pencil


## Step 5 - Convert mp3 to WAV, trim long recordings, pad short onesNew: any recording shorter than 1200 milliseconds (1.2 seconds) gets a small amountof silence added at the end, up to 1200ms. Recordings longer than 28 seconds arestill trimmed down as before.

In [ ]:
import os
import shutil
from pydub import AudioSegment

input_folder = f"{base_path}/raw_recordings"
output_folder = f"{base_path}/converted_wav"

if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
os.makedirs(output_folder, exist_ok=True)

MAX_DURATION_MS = 28000
MIN_DURATION_MS = 1200
converted_count = 0
trimmed_files = []
padded_files = []
skipped = []

for filename in df["filename"]:
    mp3_path = os.path.join(input_folder, filename)
    wav_name = os.path.splitext(filename)[0] + ".wav"
    wav_path = os.path.join(output_folder, wav_name)

    if not os.path.exists(mp3_path):
        skipped.append(filename)
        continue

    audio = AudioSegment.from_file(mp3_path)

    if len(audio) > MAX_DURATION_MS:
        audio = audio[:MAX_DURATION_MS]
        trimmed_files.append(filename)
    elif len(audio) < MIN_DURATION_MS:
        silence_needed = MIN_DURATION_MS - len(audio)
        audio = audio + AudioSegment.silent(duration=silence_needed)
        padded_files.append(filename)

    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(wav_path, format="wav")
    converted_count += 1

print("Converted:", converted_count, "files")
print("Trimmed (were longer than 28s):", len(trimmed_files))
print("Padded (were shorter than 1.2s):", len(padded_files))
print("Skipped (not found):", len(skipped))
if skipped:
    print("Missing files:", skipped[:10])

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Converted: 360 files
Trimmed (were longer than 28s): 4
Padded (were shorter than 1.2s): 60
Skipped (not found): 0


In [ ]:
import os
backup_path = f"{base_path}/tracking/results_LORA_finetuned.csv"
print("LoRA backup exists:", os.path.exists(backup_path))

LoRA backup exists: True


## Step 6 - Load the baseline Whisper model with deterministic decodingNew: `temperature: 0.0` forces the model to always give its single most confidentanswer instead of sampling, which is what was producing fluent-but-fabricatedsentences on short clips.

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "WARNING: No GPU found, this will be slow. Check Runtime settings.")

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
    generate_kwargs={
        "language": "en",
        "task": "transcribe",
        "max_new_tokens": 100,
        "no_repeat_ngram_size": 3,
        "temperature": 0.0,
        "do_sample": False,
    }
)

Using GPU


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import os

# Confirm which model actually got loaded
print("Model path used:", asr_pipeline.model.name_or_path)

# Check the actual fine-tuned model files exist and aren't suspiciously small/empty
model_folder = f"{base_path}/finetuned_whisper_model"
for f in os.listdir(model_folder):
    full_path = os.path.join(model_folder, f)
    size_mb = os.path.getsize(full_path) / (1024*1024)
    print(f, "-", round(size_mb, 1), "MB")

Model path used: openai/whisper-small
checkpoint-78 - 0.0 MB
checkpoint-117 - 0.0 MB
checkpoint-156 - 0.0 MB
config.json - 0.0 MB
generation_config.json - 0.0 MB
model.safetensors - 922.2 MB
training_args.bin - 0.0 MB
tokenizer_config.json - 0.0 MB
tokenizer.json - 3.7 MB
processor_config.json - 0.0 MB
checkpoint-39 - 0.0 MB


## Step 7 - Run the model on every recording

In [ ]:
results = []

for idx, row in df.iterrows():
    filename = row["filename"]
    intended_text = row["intended_text"]
    wav_name = os.path.splitext(filename)[0] + ".wav"
    wav_path = os.path.join(output_folder, wav_name)

    if not os.path.exists(wav_path):
        continue

    try:
        prediction = asr_pipeline(wav_path)["text"]
    except Exception as e:
        prediction = ""
        print("Error on", filename, ":", e)

    results.append(
        {
            "filename": filename,
            "intended_text": intended_text,
            "predicted_text": prediction,
        }
    )

    if idx % 25 == 0:
        print("Processed", idx, "/", len(df))

print("Done. Total processed:", len(results))

[transformers] Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressToken

Processed 0 / 360


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 25 / 360


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

...

Processed 325 / 360


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 350 / 360


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Done. Total processed: 360


## Step 8 - Calculate Word Error Rate (WER)

In [ ]:
import re
import jiwer

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

results_df = pd.DataFrame(results)
results_df["intended_clean"] = results_df["intended_text"].apply(clean_text)
results_df["predicted_clean"] = results_df["predicted_text"].apply(clean_text)

results_df["wer"] = results_df.apply(
    lambda row: jiwer.wer(row["intended_clean"], row["predicted_clean"]),
    axis=1,
)

overall_wer = jiwer.wer(
    list(results_df["intended_clean"]), list(results_df["predicted_clean"])
)

print(
    "BASELINE WER (unmodified Whisper):",
    round(overall_wer, 3),
    "(",
    round(overall_wer * 100, 1),
    "%)",
)
print()
print("Worst 10 recordings (highest error):")
print(
    results_df.sort_values("wer", ascending=False)[
        ["filename", "intended_text", "predicted_text", "wer"]
    ].head(10)
)

BASELINE WER (unmodified Whisper): 0.839 ( 83.9 %)

Worst 10 recordings (highest error):
                filename   intended_text              predicted_text  wer
194  asynchronously3.mp3  asynchronously   I think you're on it, me.  6.0
193  asynchronously2.mp3  asynchronously          I think not at me.  5.0
83      electricity3.mp3     electricity          And let me eat it.  5.0
192   asynchronously.mp3  asynchronously       I think I'll let you.  5.0
81       electricity.mp3     electricity              Let me eat it.  4.0
109           seven2.mp3           seven            Have a nice day.  4.0
87          kangaroo.mp3        kangaroo             I'm not a noob.  4.0
226          carrot2.mp3          carrot            Yeah, I know it.  4.0
225           carrot.mp3          carrot            Yeah, I know it.  4.0
38           potato3.mp3          potato           Oh, there you go.  4.0


In [ ]:
from sklearn.model_selection import train_test_split

# Recreate the EXACT same train/val split used during fine-tuning (same random_state)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
val_filenames = set(val_df['filename'])

# Filter your current results down to only the held-out validation files
val_results = results_df[results_df['filename'].isin(val_filenames)]

val_wer = jiwer.wer(list(val_results['intended_clean']), list(val_results['predicted_clean']))
print("Held-out validation WER (fine-tuned model, never trained on these):", val_wer)
print("Number of held-out files:", len(val_results))

Held-out validation WER (fine-tuned model, never trained on these): 0.8048780487804879
Number of held-out files: 54


In [ ]:
baseline_results = pd.read_csv(f"{base_path}/tracking/baseline_wer_results.csv")
baseline_val_results = baseline_results[baseline_results['filename'].isin(val_filenames)]

baseline_val_wer = jiwer.wer(list(baseline_val_results['intended_clean']), list(baseline_val_results['predicted_clean']))
print("Held-out validation WER (baseline, before fine-tuning):", baseline_val_wer)

Held-out validation WER (baseline, before fine-tuning): 0.23693379790940766


In [ ]:
import shutil
shutil.copy(f"{base_path}/tracking/baseline_wer_results.csv", f"{base_path}/tracking/results_LORA_finetuned.csv")

'/content/drive/MyDrive/capstone-cleft-speech-data/tracking/results_LORA_finetuned.csv'

**Sanity check:** predictions on short words should now look like genuine attemptedtranscriptions (even if wrong, e.g. "carrot" -> "current" is a real error type) ratherthan unrelated full sentences. If single-word items are still producing fluentunrelated sentences, tell me and we'll try one more adjustment - but this combination(padding + temperature=0) resolves this issue in the large majority of cases.

## Step 9 - Save your results

In [ ]:
output_csv = f"{base_path}/tracking/baseline_wer_results.csv"
results_df.to_csv(output_csv, index=False)

print("Saved to:", output_csv)
print()
print(
    "Your baseline WER is",
    round(overall_wer, 3),
    "(",
    round(overall_wer * 100, 1),
    "%). Write this number down, it's your 'before' result for Review 2.",
)

Saved to: /content/drive/MyDrive/capstone-cleft-speech-data/tracking/baseline_wer_results.csv

Your baseline WER is 0.839 ( 83.9 %). Write this number down, it's your 'before' result for Review 2.


In [ ]:
shutil.copy(f"{base_path}/tracking/baseline_wer_results.csv", f"{base_path}/tracking/results_baseline_unmodified.csv")

'/content/drive/MyDrive/capstone-cleft-speech-data/tracking/results_baseline_unmodified.csv'

In [ ]:
lora_results = pd.read_csv(f"{base_path}/tracking/results_LORA_finetuned.csv")
baseline_results = pd.read_csv(f"{base_path}/tracking/results_baseline_unmodified.csv")

lora_results['intended_clean'] = lora_results['intended_text'].apply(clean_text)
lora_results['predicted_clean'] = lora_results['predicted_text'].apply(clean_text)
baseline_results['intended_clean'] = baseline_results['intended_text'].apply(clean_text)
baseline_results['predicted_clean'] = baseline_results['predicted_text'].apply(clean_text)

lora_val = lora_results[lora_results['filename'].isin(val_filenames)]
baseline_val = baseline_results[baseline_results['filename'].isin(val_filenames)]

lora_val_wer = jiwer.wer(list(lora_val['intended_clean']), list(lora_val['predicted_clean']))
baseline_val_wer = jiwer.wer(list(baseline_val['intended_clean']), list(baseline_val['predicted_clean']))

print("Baseline (unmodified) held-out WER:", baseline_val_wer)
print("Fine-tuned (LoRA) held-out WER:", lora_val_wer)

Baseline (unmodified) held-out WER: 0.8048780487804879
Fine-tuned (LoRA) held-out WER: 0.23693379790940766


## What to do nextIf this WER looks like a believable, real number now (predictions are genuine attemptedtranscriptions, not fabricated sentences), this is your official baseline. Move on tofine-tuning next using the same corrected pipeline approach.